# **Annotating an existing NWB file with HERD ontology references**

This tutorial shows how to add machine-readable ontology references to an NWB file that is **already published** on DANDI, using a session from [DANDI:001528](https://dandiarchive.org/dandiset/001528/draft).

## Why

NWB stores many metadata values as free text: the subject's species (`"Mus musculus"`), its strain (`"C57BL/6J"`), and the brain region a virus was injected into or a fiber was implanted in (`"SNr"`, `"PPN"`). A person reads these fine, but a program cannot tell whether `"SNr"` in one file and `"substantia nigra pars reticulata"` in another are the same structure.

[HERD](https://hdmf.readthedocs.io/en/stable/tutorials/plot_external_resources.html) (HDMF External Resources Data) fixes this by storing, inside the file under `/general/external_resources`, a link from each of those strings to a term in a standard ontology:

| Field | Value in the file | Ontology term |
|---|---|---|
| `Subject.species` | `Mus musculus` | [NCBITaxon:10090](http://purl.obolibrary.org/obo/NCBITaxon_10090) |
| `Subject.strain` | `C57BL/6J` | [RRID:IMSR_JAX:000664](https://scicrunch.org/resolver/RRID:IMSR_JAX:000664) |
| `location` | `SNr` | [MBA:381](https://purl.brain-bican.org/ontology/mbao/MBA_381) (Allen Mouse Brain Atlas) |

The original values are not changed; HERD only adds references next to them.

## What this notebook does

1. Download one NWB file from DANDI:001528.
2. Look at the free-text values that can be annotated.
3. Let NeuroConv propose ontology terms for them (`infer_*` functions).
4. Review the proposal and add the terms NeuroConv did not recognize.
5. Write the terms into the file as HERD references (`add_*_external_resource(s)` functions).
6. Read the file back and check the references.

## Requirements

The ontology tools are not in a NeuroConv release yet. Install NeuroConv from the development branch into this environment:

```bash
pip install "neuroconv @ git+https://github.com/catalystneuro/neuroconv.git@add-anatomy-herd-ontology"
```

---

# 1. Setup

In [1]:
from pathlib import Path

# The file uses these NWB extensions. Importing them registers their specs with PyNWB, which
# HERD needs in order to annotate objects defined by an extension (here, `ViralVectorInjection`).
import ndx_fiber_photometry  # noqa: F401
import ndx_ophys_devices  # noqa: F401
import ndx_optogenetics  # noqa: F401
import pandas as pd
from dandi.dandiapi import DandiAPIClient
from hdmf.common import DynamicTable
from pynwb import NWBHDF5IO

from neuroconv.tools.ontology import (
    add_brain_region_external_resources,
    add_species_external_resource,
    add_strain_external_resource,
    get_brain_region_term,
    infer_brain_region_ontology_metadata,
    infer_species_ontology_metadata,
    infer_strain_ontology_metadata,
)

pd.set_option("display.max_colwidth", None)

# 2. Download the file from DANDI

HERD references are written **into** the file, so we need a local copy opened in read/write mode. (Streaming the file from DANDI, as in the `Lotfi_2025.ipynb` tutorial, is read-only.)

The dandiset is public, so no DANDI API key is needed.

In [2]:
# Session
dandiset_id = "001528"
subject_id = "C4561"  # Example subject
session_id = "varying-durations"  # Varying-duration optogenetic stimulation session

# Where to save the file (NWB files are git-ignored in this repository)
download_folder = Path("./nwb_files")

In [3]:
def download_nwb_from_dandi(dandiset_id, subject_id, session_id, download_folder):
    '''Download one NWB file from DANDI, skipping the download if it is already there.'''
    pattern = f"sub-{subject_id}/sub-{subject_id}_ses-{session_id}*.nwb"

    with DandiAPIClient() as client:
        assets = list(client.get_dandiset(dandiset_id, "draft").get_assets_by_glob(pattern=pattern, order="path"))
        if len(assets) != 1:
            raise ValueError(f"Expected 1 file, found {len(assets)} for pattern {pattern}")
        asset = assets[0]

        nwbfile_path = Path(download_folder) / Path(asset.path).name
        if not nwbfile_path.exists():
            nwbfile_path.parent.mkdir(parents=True, exist_ok=True)
            asset.download(nwbfile_path)

    return nwbfile_path


nwbfile_path = download_nwb_from_dandi(dandiset_id, subject_id, session_id, download_folder)
nwbfile_path

WindowsPath('nwb_files/sub-C4561_ses-varying-durations_image.nwb')

# 3. Look at the values that can be annotated

NeuroConv currently annotates four kinds of free-text values:

- `Subject.species`
- `Subject.strain`
- every anatomical `location` field: electrode groups and the electrodes table, imaging planes, intracellular electrodes, optogenetic stimulus sites, viral vector injections, and the `FiberPhotometryTable`
- pose-estimation skeleton node names (`ndx-pose`), which this dataset does not have

Let's see what this file contains.

In [4]:
with NWBHDF5IO(nwbfile_path, mode="r", load_namespaces=True) as io:
    nwbfile = io.read()

    print("Subject.species:", repr(nwbfile.subject.species))
    print("Subject.strain: ", repr(nwbfile.subject.strain))
    print()
    print("Viral vector injections:")
    for obj in nwbfile.objects.values():
        if getattr(obj, "neurodata_type", None) == "ViralVectorInjection":
            print(f"  {obj.name}: location={obj.location!r}")
    print()
    fiber_photometry_table = nwbfile.lab_meta_data["fiber_photometry"].fiber_photometry_table
    print("FiberPhotometryTable locations:", sorted(set(fiber_photometry_table["location"][:])))
    print()
    print("Existing HERD:", nwbfile.external_resources)

Subject.species: 'Mus musculus'
Subject.strain:  'C57BL/6J'

Viral vector injections:
  viral_vector_injection_PPN: location='PPN'
  viral_vector_injection_SNr: location='SNr'

FiberPhotometryTable locations: ['SNr']

Existing HERD: None


c:\Users\amtra\anaconda3\envs\dev-env\Lib\site-packages\hdmf\spec\namespace.py:772: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
ndx-fiber-photometry - cached version: 0.2.2, loaded version: 0.2.3
ndx-optogenetics - cached version: 0.3.0, loaded version: 0.4.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


The file has no HERD yet (`None`). The values to annotate are the species, the strain, and two brain regions: `SNr` (substantia nigra, reticular part, where GCaMP was expressed and the fiber was placed) and `PPN` (pedunculopontine nucleus, where ChrimsonR was injected for optogenetic stimulation).

# 4. Let NeuroConv propose ontology terms

The `infer_*` functions resolve each free-text value to an ontology term and write it into a plain `metadata` dictionary under `metadata["ontology"]`, keyed by the exact string found in the file. Nothing is written to the file at this step.

- Species are resolved against NCBI Taxonomy, strains against RRIDs.
- Brain regions are resolved against an atlas chosen from the subject's species: the Allen Mouse Brain Atlas (MBA) here, because the subject is a mouse.

In [5]:
with NWBHDF5IO(nwbfile_path, mode="r", load_namespaces=True) as io:
    nwbfile = io.read()

    metadata = {"Subject": {"species": nwbfile.subject.species, "strain": nwbfile.subject.strain}}
    infer_species_ontology_metadata(metadata)
    infer_strain_ontology_metadata(metadata)
    infer_brain_region_ontology_metadata(nwbfile, metadata)

metadata["ontology"]

{'species': {'Mus musculus': {'id': 'NCBITaxon:10090',
   'uri': 'http://purl.obolibrary.org/obo/NCBITaxon_10090'}},
 'strain': {'C57BL/6J': {'id': 'RRID:IMSR_JAX:000664',
   'uri': 'https://scicrunch.org/resolver/RRID:IMSR_JAX:000664'}},
 'brain_regions': {'SNr': {'id': 'MBA:381',
   'uri': 'https://purl.brain-bican.org/ontology/mbao/MBA_381'}}}

# 5. Review the proposal and fill the gaps

Species, strain and `SNr` were recognized. `PPN` is missing: NeuroConv ships a curated subset of the Allen atlas, and the pedunculopontine nucleus is not in it.

In [6]:
print("SNr ->", get_brain_region_term("SNr", species="Mus musculus"))
print("PPN ->", get_brain_region_term("PPN", species="Mus musculus"))

SNr -> BrainRegionTerm(acronym='SNr', name='Substantia nigra, reticular part', curie='MBA:381', entity_uri='https://purl.brain-bican.org/ontology/mbao/MBA_381')
PPN -> None


We can add the missing term ourselves. The Allen Mouse Brain Atlas lists the pedunculopontine nucleus with acronym `PPN` and structure ID [1052](https://api.brain-map.org/api/v2/data/Structure/1052.json), so its term is `MBA:1052`.

Every term is a `{"id": <CURIE>, "uri": <resolvable URI>}` dictionary. A term you add by hand is never overwritten by the `infer_*` functions, so the same approach also works for correcting a proposed term.

In [7]:
metadata["ontology"]["brain_regions"]["PPN"] = {
    "id": "MBA:1052",
    "uri": "https://purl.brain-bican.org/ontology/mbao/MBA_1052",
}

metadata["ontology"]

{'species': {'Mus musculus': {'id': 'NCBITaxon:10090',
   'uri': 'http://purl.obolibrary.org/obo/NCBITaxon_10090'}},
 'strain': {'C57BL/6J': {'id': 'RRID:IMSR_JAX:000664',
   'uri': 'https://scicrunch.org/resolver/RRID:IMSR_JAX:000664'}},
 'brain_regions': {'SNr': {'id': 'MBA:381',
   'uri': 'https://purl.brain-bican.org/ontology/mbao/MBA_381'},
  'PPN': {'id': 'MBA:1052',
   'uri': 'https://purl.brain-bican.org/ontology/mbao/MBA_1052'}}}

# 6. Write the HERD references into the file

The `add_*` functions are deterministic: they take the terms stated in `metadata["ontology"]` and attach them to every matching value in the file. Values that `metadata` does not mention are left alone.

We open the file in `"r+"` mode, add the references, and write the changes back. Only the new `/general/external_resources` group is added; the existing data are not rewritten.

In [8]:
with NWBHDF5IO(nwbfile_path, mode="r+", load_namespaces=True) as io:
    nwbfile = io.read()

    species_added = add_species_external_resource(nwbfile, metadata=metadata)
    strain_added = add_strain_external_resource(nwbfile, metadata=metadata)
    number_of_brain_region_references = add_brain_region_external_resources(nwbfile, metadata=metadata)

    io.write(nwbfile)

print("Species annotated:", species_added)
print("Strain annotated: ", strain_added)
print("Brain-region references added:", number_of_brain_region_references)

Species annotated: True
Strain annotated:  True
Brain-region references added: 3


There are three brain-region references: `SNr` appears in two places (on its `ViralVectorInjection` object and in the `location` column of the `FiberPhotometryTable`), and `PPN` in one (on its `ViralVectorInjection` object).

# 7. Read the file back and check the references

HERD is a set of tables stored in the file. `to_dataframe()` flattens them into one row per reference: which object and field carry the value, the value itself (`key`), and the ontology term it points to (`entity_id`, `entity_uri`).

In [9]:
with NWBHDF5IO(nwbfile_path, mode="r", load_namespaces=True) as io:
    nwbfile = io.read()
    references = nwbfile.external_resources.to_dataframe()

    # Label each annotated object by name; a table column is labelled "<table>/<column>"
    object_names = {
        object_id: f"{obj.parent.name}/{obj.name}" if isinstance(obj.parent, DynamicTable) else obj.name
        for object_id, obj in nwbfile.objects.items()
    }
    references["object_name"] = references["object_id"].map(object_names)

references[["object_type", "object_name", "relative_path", "key", "entity_id", "entity_uri"]]

,object_type,object_name,relative_path,key,entity_id,entity_uri
0,Subject,subject,species,Mus musculus,NCBITaxon:10090,http://purl.obolibrary.org/obo/NCBITaxon_10090
1,Subject,subject,strain,C57BL/6J,RRID:IMSR_JAX:000664,https://scicrunch.org/resolver/RRID:IMSR_JAX:000664
2,ViralVectorInjection,viral_vector_injection_PPN,location,PPN,MBA:1052,https://purl.brain-bican.org/ontology/mbao/MBA_1052
3,ViralVectorInjection,viral_vector_injection_SNr,location,SNr,MBA:381,https://purl.brain-bican.org/ontology/mbao/MBA_381
4,VectorData,fiber_photometry_table/location,,SNr,MBA:381,https://purl.brain-bican.org/ontology/mbao/MBA_381


Each row links a value in the file to its ontology term. For example, the `location` of the `viral_vector_injection_PPN` object, `PPN`, now points to `MBA:1052`.

# 8. Next steps

- **Annotate every session.** The same steps work in a loop over all assets in the dandiset. Collect the terms you add by hand (like `PPN` above) in one dictionary and reuse it for each file.
- **Annotate during conversion.** For new conversions you do not need a separate step: when `metadata["ontology"]` is filled in, NeuroConv's `run_conversion` writes the HERD references automatically.
- **Publish the annotated files.** Upload the modified files to the dandiset with `dandi upload`, which replaces the existing assets in the draft version.

See the NeuroConv [ontology user guide](https://github.com/catalystneuro/neuroconv/blob/add-anatomy-herd-ontology/docs/user_guide/ontology.rst) for the full list of functions and supported vocabularies.